This notebook builds the preprocessing pipeline for the fish sequence and phylogeny project.

The goals of this stage are:

1. Load and standardize the MitoFish sequence and taxonomy tables.
2. Combine annotation, taxonomy, and FASTA sequence information into one integrated sequence dataset.
3. Match MitoFish species to the Fish Tree of Life.
4. Resolve species name mismatches between the two resources.
5. Export a filtered dataset containing only species that occur in both sources.
6. Export phylogeny-derived files that can be used later for transformer and graph-based models.

---

# Stage 1 — MitoFish preprocessing and dataset integration

## 1. Environment setup and path definitions

This cell imports the required libraries and defines all input and output paths used throughout the notebook.

It also creates the output directory if it does not already exist, so later export steps can write their files without additional setup.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from ete3 import Tree

path_dataset = Path("datasets")
path_out_dir = Path("output")

path_fishtree = path_dataset / "fishtree"
path_fishtree_taxonomy = path_fishtree / "PFC_taxonomy.csv"
path_fishtree_tree = path_fishtree / "actinopt_12k_raxml.tre"

path_mitofish = path_dataset / "MitoFish"
path_mitofish_fasta = path_mitofish / "mitofishdb.fa"
path_mitofish_seq_annotation = path_mitofish / "seq_annotation.parquet"
path_mitofish_seq_description = path_mitofish / "seq_description.parquet"
path_mitofish_seq_heterospecific = path_mitofish / "seq_heterospecific.parquet"
path_mitofish_seq_taxonid = path_mitofish / "seq_taxonid.parquet"
path_mitofish_taxonid_name = path_mitofish / "taxonid_name.parquet"
path_mitofish_taxonid_speciesid = path_mitofish / "taxonid_speciesid.parquet"
path_mitofish_speciesid_lineageid = path_mitofish / "speciesid_lineageid.parquet"

## 2. Load MitoFish gene annotation table

This cell loads the MitoFish sequence annotation table from parquet format.

The table contains per-accession metadata such as:

- gene name
- genomic start and end positions
- strand
- sequence length

The column dtypes are standardized immediately so later joins and aggregations are more memory-efficient and consistent.

In [2]:
df_mf_seq_an = pd.read_parquet(path_mitofish_seq_annotation).astype({
    'accession': 'string',
    'gene': 'category',
    'start': 'int32',
    'end': 'int32',
    'strand': 'category',
    'length': 'int32'})
df_mf_seq_an

,accession,gene,start,end,strand,length
0,AB000667,12S_rRNA,2469,3417,1,949
1,AB000667,16S_rRNA,3491,3576,1,86
2,AB000667,Cytb,2,857,1,856
3,AB000673,ND5,194,1008,1,815
4,AB000674,COXIII,1,233,1,233
...,...,...,...,...,...,...
1004508,Z97553,Cytb,1,363,1,363
1004509,Z97554,Cytb,1,363,1,363
1004510,Z97555,Cytb,1,363,1,363
1004511,Z97556,Cytb,1,363,1,363


## 3. Load MitoFish accession-to-taxon mapping

This cell loads the table that links each sequence accession to an NCBI taxon ID.

This mapping is needed to connect sequence-level records to species-level taxonomy information in later steps.

In [3]:
df_mf_seq_taxid = pd.read_parquet(path_mitofish_seq_taxonid).astype({
    'accession': 'string',
    'create_date': 'datetime64[ns]',
    'taxon_id': 'int32'
})
df_mf_seq_taxid

,accession,create_date,taxon_id
0,AB000667,1997-02-05,8255
1,AB000668,1997-02-05,8255
2,AB000669,1997-02-05,8255
3,AB000670,1997-02-05,8255
4,AB000671,1997-02-05,8255
...,...,...,...
947202,Z97553,1999-09-20,27763
947203,Z97554,1999-09-20,27763
947204,Z97555,1999-09-20,27763
947205,Z97556,1999-09-20,27763


## 4. Load lineage ID to lineage name mapping

This cell loads the taxonomy name table, which maps lineage identifiers to their taxonomic names and ranks.

It provides the human-readable taxonomy labels that will later be merged into the species-level taxonomy table.

In [4]:
df_mf_taxid_name = pd.read_parquet(path_mitofish_taxonid_name).astype({
    'lineage_id': 'int32',
    'lineage_rank': 'category',
    'lineage_name': 'string'
})
df_mf_taxid_name

,lineage_id,lineage_rank,lineage_name
0,1000278,subspecies,Acheilognathus tabira tohokuensis
1,1000982,species,Steindachneridion melanodermatum
2,1001000,species,Sebastes chrysomelas x Sebastes carnatus
3,100124,genus,Pseudobarbus
4,100125,species,Pseudobarbus burchelli
...,...,...,...
60912,999360,species,Hypselobarbus periyarensis
60913,999361,species,Barbodes carnaticus
60914,999678,genus,Oxygaster
60915,999679,species,Oxygaster anomalura


## 5. Load taxon ID to species ID mapping

This cell loads the table that maps NCBI taxon IDs to MitoFish species IDs.

This is an intermediate key table that allows us to move from accession-level taxon assignments to species-level taxonomy.

In [5]:
df_mf_taxid_spid = pd.read_parquet(path_mitofish_taxonid_speciesid).astype({
    'taxon_id': 'int32',
    'species_id': 'int32'
})
df_mf_taxid_spid

,taxon_id,species_id
0,1000982,1000982
1,1001000,1001000
2,100125,100125
3,100126,100126
4,1001636,1001636
...,...,...
55177,999359,999359
55178,999360,999360
55179,999361,999361
55180,999679,999679


## 6. Build the species-level taxonomy table

This cell reconstructs the species taxonomy by combining:

- species ID to lineage ID relationships
- lineage ID to lineage name mappings

The result is reshaped into a wide species taxonomy table, where each species is associated with its major taxonomic ranks such as order, family, genus, and species.

This table will later be merged into the sequence annotation data so every sequence record can be linked to its taxonomy.

In [6]:
df_mf_spid_linid = pd.read_parquet(path_mitofish_speciesid_lineageid).astype({
    'species_id': 'int32',
    'rank_order': 'int32',
    'rank_name': 'category',
    'lineage_id': 'int32'
})

df_mf_spid_tax = pd.merge(df_mf_spid_linid, df_mf_taxid_name, on="lineage_id")

target_ranks = ['order', 'family', 'genus', 'species']
df_mf_spid_tax = df_mf_spid_tax[df_mf_spid_tax['rank_name'].isin(target_ranks)]

df_mf_spid_tax_piv = df_mf_spid_tax.pivot(
    index='species_id',
    columns='rank_name',
    values='lineage_name'
).reset_index()

df_mf_spid_tax_piv

rank_name,species_id,species,family,genus,order
0,7748,Lampetra fluviatilis,Petromyzontidae,Lampetra,Petromyzontiformes
1,7750,Lampetra planeri,Petromyzontidae,Lampetra,Petromyzontiformes
2,7752,Lampetra aepyptera,Petromyzontidae,Lampetra,Petromyzontiformes
3,7753,Lethenteron reissneri,Petromyzontidae,Lethenteron,Petromyzontiformes
4,7755,Mordacia mordax,Petromyzontidae,Mordacia,Petromyzontiformes
...,...,...,...,...,...
54392,3683097,Ecsenius sp. T76863-1,Blenniidae,Ecsenius,Blenniiformes
54393,3683777,Elates sp. T77057-1,Platycephalidae,Elates,Perciformes
54394,3684156,Danio rerio x Danio albolineatus,Danionidae,Danio,Cypriniformes
54395,3684157,Danio rerio x Danio kyathit,Danionidae,Danio,Cypriniformes


## 7. Merge annotation and taxonomy into a sequence metadata table

This cell combines the sequence annotation table with the taxonomic mapping tables.

The output is a sequence-level metadata table in which each accession is associated with:

- its gene annotation
- sequence length
- species name
- genus
- family
- order

Columns that are only needed for intermediate joins are removed afterward to keep the table compact and easier to work with.

In [7]:
df_mf_seq_an_taxid = pd.merge(df_mf_seq_an, df_mf_seq_taxid, on="accession")
df_mf_seq_an_spid = pd.merge(df_mf_seq_an_taxid, df_mf_taxid_spid, on="taxon_id")
df_mf_seq_an_tax = pd.merge(df_mf_seq_an_spid, df_mf_spid_tax_piv, on='species_id')
df_mf_seq_an_tax.drop(columns=['create_date', 'taxon_id', 'species_id', 'start', 'end', 'strand'], inplace=True)

df_mf_seq_an_tax

,accession,gene,length,species,family,genus,order
0,AB000667,12S_rRNA,949,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
1,AB000667,16S_rRNA,86,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
2,AB000667,Cytb,856,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
3,AB000673,ND5,815,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
4,AB000674,COXIII,233,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
...,...,...,...,...,...,...,...
1004499,Z97553,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004500,Z97554,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004501,Z97555,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004502,Z97556,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes


## 8. Load and parse the MitoFish FASTA sequences

This cell reads the raw FASTA file and converts it into a tabular dataframe with:

- accession
- nucleotide sequence

The parser also:

- strips whitespace
- converts sequences to uppercase
- handles headers that contain multiple accession identifiers by splitting and exploding them into separate rows

This produces a clean sequence table that can be merged with the metadata table from the previous step.

In [8]:
df_mf_fasta = pd.read_csv(path_mitofish_fasta, lineterminator='>', header=None)
df_mf_fasta = df_mf_fasta[0].str.split('\n', expand=True, n=1).set_axis(["accession", "sequence"], axis=1)
df_mf_fasta['accession'] = df_mf_fasta['accession'].str.strip()
df_mf_fasta['sequence'] = df_mf_fasta['sequence'].str.upper().str.strip()

df_mf_fasta['accession'] = df_mf_fasta['accession'].str.split(';')
df_mf_fasta = df_mf_fasta.explode('accession')

df_mf_fasta = df_mf_fasta.astype({
    'accession': 'string',
    'sequence': 'string'
})

df_mf_fasta

,accession,sequence
0,PQ550629,GGCACTGCTTTAAGCCTTCTTATTCGAGCAGAACTAAGCCAACCAG...
1,MN473720,ACTAAATCAGCCCTCGTCCATAAACCCATATCACAGGACTGAACAC...
2,GU593119,GTGATGAAACTTCGGATCCCTCCTACTACTCTGTCTCTCAGCACAA...
3,MH715312,TGGGATTAGATACCCCACTATGCTTAGTCGTAAACCTCGGTAGGCC...
4,OM978181,GGTGCCTGAGCCGGAATAGTAGGTACAGCCCTGAGCCTGCTAATTC...
...,...,...
580146,GU210318,CGCCAGACTACGAGCATAGCTTAAAACCCAAAGGACTTGGCGGTGC...
580147,MZ723175,CTAGTATTCGGTGCATGAGCTGGAATAGTTGGCACGGCCTTAAGCT...
580148,KF808193,CCTTTATTTAATCTTTGGTGCATGAGCGGGGATAGTGGGTACTGGT...
580149,KF021391,ATGCCTCAACTTAACCCCGCACCCTGGTTCCTAATTATAGTATTCT...


## 9. Inspect the sequence alphabet

This quick check collects all unique characters present in the sequence strings.

It is useful for verifying that the FASTA data only contains the expected nucleotide symbols and for spotting ambiguous or invalid characters before downstream modeling.

In [9]:
set().union(*df_mf_fasta['sequence'].unique())

{'A', 'B', 'C', 'D', 'G', 'H', 'K', 'M', 'N', 'R', 'S', 'T', 'V', 'W', 'Y'}

## 10. Build the complete MitoFish sequence dataset

This cell merges the sequence metadata and FASTA tables into one integrated dataset containing:

- accession
- gene
- taxonomy
- sequence
- length

Duplicate rows are removed at the level of `(sequence, gene, species)` so that identical gene sequences from the same species are not counted multiple times.

A short summary is then shown to inspect:

- how many species are present
- how many rows each species contributes

In [10]:
df_mf_all_genes = pd.merge(df_mf_seq_an_tax, df_mf_fasta, on="accession")
df_mf_all_genes.drop_duplicates(subset=['sequence', 'gene', 'species'], inplace=True)

print(f"Number of species in MitoFish: {df_mf_all_genes['species'].nunique()}")
df_mf_all_genes.value_counts(subset='species').to_frame()

Number of species in MitoFish: 43808


,count
species,
Amblyraja radiata,6715
Gadus morhua,4710
Chaenogobius annularis,3163
Carcharodon carcharias,2703
Carcharhinus leucas,2594
...,...
Rhabdoblennius ellipes,1
Entomacrodus cadenati,1
Myxodagnus opercularis,1


# Stage 2 — Load and clean the Fish Tree of Life species list

## 11. Parse the Fish Tree of Life leaf names

This cell loads the Fish Tree of Life Newick file and extracts all leaf names as species labels.

Several cleanup steps are applied:

- underscores are replaced with spaces
- duplicated species epithets are collapsed
- non-specific placeholder names such as `sp.`, `cf.`, `aff.`, and related patterns are removed

The goal is to obtain a cleaner species list from the tree that can be compared against the MitoFish species names.

In [11]:
t = Tree(str(path_fishtree_tree))
t_leaves = t.get_leaves()
df_leaves = pd.DataFrame([l.name.replace("_", " ") for l in t_leaves], columns=['species'], dtype='string')
df_leaves['species'] = df_leaves['species'].str.replace(r'\b(\w+) \1\b', r'\1', regex=True)

BAD_TOKENS = (" sp", " sp.", " spp", " spp.", " cf", " cf.", " aff", " aff.", " nr", " nr.")

df_leaves = df_leaves[~df_leaves['species'].str.endswith(BAD_TOKENS)]

print(f"Number of species in The Fish Tree of Life: {df_leaves['species'].nunique()}")
df_leaves

Number of species in The Fish Tree of Life: 11629


,species
0,Gambusia marshi
1,Gambusia panuco
2,Gambusia regani
3,Gambusia aurata
4,Gambusia hurtadoi
...,...
11633,Polypterus congicus
11634,Polypterus retropinnis
11635,Polypterus mokelembembe
11636,Polypterus ornatipinnis


## 12. Identify tree species that are missing from MitoFish

This cell compares the Fish Tree of Life leaf names against the species present in the MitoFish dataset.

Species that occur in the tree but not in MitoFish are collected into a separate table.

To support manual inspection and external name-reconciliation tools, the missing names are also exported in chunks of 100 species per text file.

In [12]:
df_leaves_not_in_mf = df_leaves[~df_leaves['species'].isin(df_mf_all_genes['species'])]

path_leaves_not_in_mf =  path_fishtree / "leaves_not_in_mf"
path_leaves_not_in_mf.mkdir(exist_ok=True)

for i in range(0, df_leaves_not_in_mf.shape[0], 100):
    df_leaves_not_in_mf.iloc[i:i+100].to_csv(path_leaves_not_in_mf / f"leaves_not_in_mf_{i}-{i+100}.txt", 
                                             index=False, header=False)

print(f"Number of Fish Tree of Life species not in MitoFish: {df_leaves_not_in_mf['species'].nunique()}")
df_leaves_not_in_mf

Number of Fish Tree of Life species not in MitoFish: 1037


,species
10,Gambusia zarskei
119,Limia dominicensis
148,Micropoecilia picta
194,Cyprinodon hubbsi
198,Cyprinodon variegatus ovinus
...,...
11527,Anguilla bengalensis labiata
11532,Anguilla bicolor pacifica
11545,Anguilla australis schmidtii
11624,Polypterus polli


## 13. Load and filter manual species remapping tables

Many species mismatches arise due to **taxonomic revisions**, such as genus changes.

To resolve these inconsistencies, the MitoFish species name mapping tool was used:

https://mitofish.aori.u-tokyo.ac.jp/species/mapping

This cell loads the manually curated species-name remapping tables from the `leaves_remap` directory.

Several filters are then applied so that only plausible remappings are kept:

- only rows marked as needing manual checking are considered
- the target NCBI name must not already exist directly in the Fish Tree of Life species list
- the target NCBI name must exist in MitoFish
- the source and target names must have a plausible species epithet match

Finally, duplicate source names are removed so that each original tree name maps to at most one replacement name.

In [13]:
path_leaves_remap = path_fishtree / "leaves_remap"

df_map = pd.concat(
    pd.read_csv(path_leaves_remap / f"table-mapping-species-names-{i}.csv") for i in range (1, 12)
)

# Only select rows with species mismatch
df_map = df_map[df_map['Need Manually Check?'] == "Yes"]
# Only select rows where Name in NCBI is not already present in Fish Tree of Life species
df_map = df_map[~df_map['Name in NCBI'].isin(df_leaves['species'])]
# Only select rows where Name in NCBI is present in MitoFish
df_map = df_map[df_map['Name in NCBI'].isin(df_mf_all_genes['species'])]
# Only select rows where the query species name matches the NCBI species name (first 3 characters)
df_map = df_map[(df_map['Query Name'].str.split().str[-1].str[:3] == df_map['Name in NCBI'].str.split().str[-1].str[:3]) |
                (df_map['Query Name'].str.split().str[-2].str[:3] == df_map['Name in NCBI'].str.split().str[-1].str[:3])]
# Drop duplicates so each query gets a single remap name
df_map.drop_duplicates('Query Name', inplace=True)

print(f"Number of Fish Tree of Life species with new names in MitoFIsh: {df_map['Query Name'].nunique()}")
df_map

Number of Fish Tree of Life species with new names in MitoFIsh: 869


,Taxon ID,Query Name,Name in NCBI,Need Manually Check?
1,154033.0,Limia dominicensis,Poecilia dominicensis,Yes
2,69234.0,Micropoecilia picta,Poecilia picta,Yes
4,2802244.0,Cyprinodon variegatus ovinus,Cyprinodon ovinus,Yes
6,30734.0,Garmanella pulchra,Jordanella pulchra,Yes
7,722643.0,Adinia xenica,Fundulus xenicus,Yes
...,...,...,...,...
36,2864443.0,Pisodonophis sangjuensis,Ophichthus sangjuensis,Yes
38,267672.0,Gnathophis nystromi ginanago,Gnathophis ginanago,Yes
39,391210.0,Poeciloconger kapala,Ariosoma kapala,Yes
41,458554.0,Bassanago albescens,Pseudoxenomystax albescens,Yes


## 14. Apply the species remapping and measure the new overlap

This cell converts the filtered remapping table into a dictionary and applies it to the Fish Tree of Life species list.

After remapping:

- duplicate names are removed
- the remapped tree species are split into those that do and do not occur in MitoFish

This gives the final post-remapping overlap between the two resources and shows which tree species still remain unmatched.

After remapping, the number of tree species without sequence data decreased from:

**1037 → 168 species**

In [14]:
name_mapping = dict(zip(df_map['Query Name'], df_map['Name in NCBI']))

df_leaves_remap = df_leaves.replace(name_mapping)
df_leaves_remap = df_leaves_remap.drop_duplicates()
df_leaves_remap_in_mf = df_leaves_remap[(df_leaves_remap['species'].isin(df_mf_all_genes['species']))]
df_leaves_remap_not_in_mf = df_leaves_remap[(~df_leaves_remap['species'].isin(df_mf_all_genes['species']))]

print(f"Number of Fish Tree of Life species not in MitoFish after name remapping: {df_leaves_remap_not_in_mf['species'].nunique()}")
df_leaves_remap_not_in_mf

Number of Fish Tree of Life species not in MitoFish after name remapping: 168


,species
10,Gambusia zarskei
194,Cyprinodon hubbsi
299,Aphanius farsicus
302,Aphanius splendens
436,Hypsolebias flammeus
...,...
11454,Thalassenchelys coheni
11532,Anguilla bicolor pacifica
11545,Anguilla australis schmidtii
11624,Polypterus polli


# Stage 3 — Build the final sequence dataset restricted to species on the tree

## 15. Keep only species shared by MitoFish and the Fish Tree of Life

This cell merges the integrated MitoFish sequence dataset with the remapped tree species list.

The result is the main modeling dataset: only gene sequences from species that can be placed on the phylogeny are retained.

The cell also computes per-gene summary statistics, including:

- number of species containing the gene
- total sequence count
- average and standard deviation of gene length
- average and standard deviation of copy count per species

These summaries are useful for understanding gene coverage and redundancy in the final dataset.

In [15]:
df_mf_all_genes_on_tree = pd.merge(df_mf_all_genes, df_leaves_remap, on='species')

gene_stats = df_mf_all_genes_on_tree.groupby('gene', observed=True).agg(
    species_with_gene=('species', 'nunique'),
    sequence_count=('gene', 'size'),
    average_length=('length', 'mean'),
    std_length=('length', 'std'),
)

counts = df_mf_all_genes_on_tree.groupby(['species', 'gene'], observed=True).size().reset_index(name='copy_count')
copy_stats = counts.groupby('gene', observed=True)['copy_count'].agg(
    avg_copies_per_species='mean',
    std_copies_per_species='std'
)

gene_stats = gene_stats.join(copy_stats)

print(f"Number of species in both Fish Tree of Life and MitoFish after name remapping: {df_mf_all_genes_on_tree['species'].nunique()}")
gene_stats

Number of species in both Fish Tree of Life and MitoFish after name remapping: 11460


,species_with_gene,sequence_count,average_length,std_length,avg_copies_per_species,std_copies_per_species
gene,,,,,,
12S_rRNA,8013,32928,608.054543,324.348389,4.109322,8.625670
16S_rRNA,8711,39029,882.816931,558.050160,4.480427,9.582049
ATPase6,4742,17894,654.183805,80.512734,3.773513,11.365315
ATPase8,4598,16086,161.304986,23.489461,3.498478,10.648001
COXI,9865,141818,705.615401,273.364191,14.375874,27.309449
COXII,4139,10666,684.679074,44.233155,2.576951,8.156224
COXIII,4227,11165,731.707031,183.029114,2.641353,8.042603
Cytb,9085,109577,921.706727,278.563210,12.061310,36.775460
ND1,4498,13486,952.976865,113.680675,2.998221,10.939134


## 16. Summarize dataset coverage by taxonomic order

This cell aggregates the shared dataset at the taxonomic order level.

For each order, it reports:

- number of species
- total number of rows
- average number of unique genes per species
- standard deviation of unique gene counts
- fraction of species with at least two genes

This provides a high-level view of which clades are best represented and can help guide later sampling or experimental design decisions.

In [16]:
# 1. Basic counts per order
order_counts = df_mf_all_genes_on_tree.groupby('order', observed=True).agg(
    num_species=('species', 'nunique'),
    num_rows=('species', 'size')
)

# 2. Count unique genes per individual species (keeping the order context)
species_gene_counts = (
    df_mf_all_genes_on_tree
    .groupby(['order', 'species'], observed=True)['gene']
    .nunique()
    .reset_index(name='unique_gene_count')
)

# 3. Aggregate species-level data up to the order level
order_gene_stats = (
    species_gene_counts.groupby('order', observed=True)['unique_gene_count']
    .agg(
        avg_unique_genes_per_species='mean',
        std_unique_genes_per_species='std', # Added standard deviation
        frac_species_with_2plus_genes=lambda x: (x >= 2).mean()
    )
)

# 4. Join, sort, and display
final_order_stats = (
    order_counts.join(order_gene_stats)
    .sort_values(by='num_species', ascending=False)
)

final_order_stats.head(30)


,num_species,num_rows,avg_unique_genes_per_species,std_unique_genes_per_species,frac_species_with_2plus_genes
order,,,,,
Cypriniformes,1672,93180,9.354067,6.088043,0.923445
Perciformes,1252,50684,7.889776,5.848664,0.924920
Siluriformes,1136,24386,5.269366,4.801209,0.858275
Cichliformes,724,12150,4.723757,4.535514,0.835635
Cyprinodontiformes,580,12989,5.096552,4.228898,0.931034
Gobiiformes,511,24991,7.113503,5.429299,0.937378
Characiformes,481,11200,4.812890,4.259587,0.956341
Labriformes,338,8683,6.375740,5.243416,0.940828
Blenniiformes,265,4357,4.535849,4.796586,0.758491


# Stage 4 — Export the processed sequence dataset

## 17. Save the final sequence-level dataset

This cell converts the main taxonomy columns to categorical dtype and writes the final sequence-level dataset to parquet format.

This exported file is the primary input for downstream model training, since it contains the cleaned and filtered MitoFish records restricted to species present in the phylogeny.

In [17]:
path_out_dir = Path("output")
path_out_dir.mkdir(exist_ok=True)

path_processed_dataset = path_out_dir / "processed_mitofish.parquet"

df_final = df_mf_all_genes_on_tree.astype({tax : 'category' for tax in ['order', 'family', 'genus', 'species'] })
df_final.to_parquet(path_processed_dataset)

df_final


,accession,gene,length,species,family,genus,order,sequence
0,AB000667,12S_rRNA,949,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
1,AB000667,16S_rRNA,86,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
2,AB000667,Cytb,856,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
3,AB000673,ND5,815,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CATTAGATTGTGATTCTAAAAACAGAGGTTGAAGCCCTCTTGCCCA...
4,AB000674,COXIII,233,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCCTTCACTATTGCAGATGGGGTTTATGGCGCCACATTCTTCGTTG...
...,...,...,...,...,...,...,...,...
473132,Z97552,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
473133,Z97553,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
473134,Z97554,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
473135,Z97555,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,NNNNNNNNNNNNNNNGTTGATCTCCCAGCTCCTTCAAACATTTCCG...


## 18. Save a species-level taxonomy and gene-count table

This cell creates a second export focused on species-level metadata.

For each species, it stores:

- species, genus, family, and order
- copy counts for each gene
- number of unique genes present
- total number of sequences across all genes

This table is useful for downstream bookkeeping, species indexing, exploratory analysis, and model-side metadata lookup.

In [18]:
path_processed_dataset_taxonomy = path_out_dir / "processed_mitofish_species.csv"

taxonomy_cols = ['species', 'genus', 'family', 'order']

df_taxonomy = df_final[taxonomy_cols].drop_duplicates()
df_taxonomy = df_taxonomy.sort_values(by='species', ignore_index=True)

gene_counts = (
    df_final.groupby(['species', 'gene'], observed=True)
    .size()
    .unstack(fill_value=0)
)

final_table = pd.merge(
    df_taxonomy, 
    gene_counts, 
    on='species', 
    how='left'
)

# 1. Identify the gene columns (all columns except the taxonomy ones)
gene_cols = [col for col in final_table.columns if col not in taxonomy_cols]

# 2. Count how many genes have a value > 0 for each row
final_table['unique_gene_count'] = (final_table[gene_cols] > 0).sum(axis=1)

# 3. (Optional) If you also want the total number of sequences (sum of all copies):
final_table['total_sequence_count'] = final_table[gene_cols].sum(axis=1)

final_table.to_csv(path_processed_dataset_taxonomy)

final_table

,species,genus,family,order,12S_rRNA,16S_rRNA,ATPase6,ATPase8,COXI,COXII,...,Cytb,ND1,ND2,ND3,ND4,ND4L,ND5,ND6,unique_gene_count,total_sequence_count
0,Abalistes stellaris,Abalistes,Balistidae,Tetraodontiformes,4,6,2,2,7,2,...,3,2,2,2,2,2,2,2,15,42
1,Abalistes stellatus,Abalistes,Balistidae,Tetraodontiformes,5,2,0,0,24,0,...,0,0,0,0,0,0,0,0,3,31
2,Abantennarius bermudensis,Abantennarius,Antennariidae,Lophiiformes,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,1
3,Abantennarius coccineus,Abantennarius,Antennariidae,Lophiiformes,5,4,2,0,24,2,...,2,2,2,2,2,2,2,2,14,55
4,Abantennarius nummifer,Abantennarius,Antennariidae,Lophiiformes,2,3,1,0,5,1,...,2,1,1,1,1,1,1,1,14,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11455,Zosterisessor ophiocephalus,Zosterisessor,Gobiidae,Gobiiformes,4,4,0,0,9,0,...,7,1,1,0,0,0,0,0,6,26
11456,Zu cristatus,Zu,Trachipteridae,Lampriformes,5,3,2,2,8,2,...,2,2,2,2,2,2,2,2,15,40
11457,Zu elongatus,Zu,Trachipteridae,Lampriformes,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,1
11458,Zungaro jahu,Zungaro,Pimelodidae,Siluriformes,1,0,2,0,6,0,...,0,0,0,0,0,0,0,0,3,9


# Stage 5 — Prune the Fish Tree of Life to the processed species set

## 19. Create a pruned phylogeny containing only shared species

This cell copies the original Fish Tree of Life and keeps only the leaves whose species are present in the remapped overlap with MitoFish.

During pruning:

- leaf names are updated to their remapped species names
- all other leaves are removed
- original branch lengths are preserved

The resulting pruned tree is exported as a Newick file and will serve as the phylogenetic backbone for later representation learning.

In [19]:
pruned_t = t.copy()

# Collect only the leaves that exist in MitoFish after remapping names
nodes_to_keep = []
for idx, node in enumerate(pruned_t.get_leaves()):
    if idx in df_leaves_remap_in_mf.index:
        node.name = df_leaves_remap_in_mf.loc[idx]['species']
        nodes_to_keep.append(node)

# This keeps only these nodes and removes all others in one go
pruned_t.prune(nodes_to_keep, preserve_branch_length=True)

path_processed_fishtree = path_out_dir / "processed_fishtree.nwk"
pruned_t.write(outfile=path_processed_fishtree)

print(f"Number of leaves before pruning: {len(t.get_leaves())}")
print(f"Number of leaves after pruning: {len(pruned_t.get_leaves())}")

Number of leaves before pruning: 11638
Number of leaves after pruning: 11460


# Stage 6 — Export pairwise phylogenetic distance matrices

## 20. Compute branch-length and topology distances between all species

This cell builds two pairwise distance matrices for the pruned tree:

1. **Branch-length distance**  
   The sum of branch lengths along the path between two species.

2. **Topology distance**  
   The number of edges along the path between two species.

The cell also exports a stable species index table so that matrix rows and columns can be matched back to species names later.

In [20]:
# ------------------------------------------------------------
# Stable leaf/species order
# ------------------------------------------------------------
species_order = sorted(pruned_t.get_leaf_names())
n_species = len(species_order)
species_to_idx = {sp: i for i, sp in enumerate(species_order)}

df_species_index = pd.DataFrame({
    "matrix_index": np.arange(n_species, dtype=np.int32),
    "species": species_order,
})
df_species_index.to_csv(path_out_dir / "processed_fishtree_species_index.csv", index=False)


# ------------------------------------------------------------
# Output matrices
# ------------------------------------------------------------
branch_dist = np.zeros((n_species, n_species), dtype=np.float32)
topology_dist = np.zeros((n_species, n_species), dtype=np.int16)


# ------------------------------------------------------------
# Postorder distance fill
#
# Return value for each node:
#   list of tuples:
#       (species_matrix_index, branch_distance_to_this_node, edge_distance_to_this_node)
# ------------------------------------------------------------
def fill_pairwise_distances_postorder(node):
    # Leaf: just return itself at distance 0 to itself
    if node.is_leaf():
        idx = species_to_idx[node.name]
        return [(idx, 0.0, 0)]

    child_lists = []

    # First recurse into children
    for child in node.children:
        child_info = fill_pairwise_distances_postorder(child)

        # Shift distances from child -> current node
        shifted = [
            (leaf_idx, branch_to_child + float(child.dist), edge_to_child + 1)
            for (leaf_idx, branch_to_child, edge_to_child) in child_info
        ]
        child_lists.append(shifted)

    # All pairs from different child subtrees have this node as LCA
    for a in range(len(child_lists)):
        left = child_lists[a]
        for b in range(a + 1, len(child_lists)):
            right = child_lists[b]

            for i_idx, i_branch, i_edges in left:
                # local aliases help a tiny bit in Python loops
                bd_row = branch_dist[i_idx]
                td_row = topology_dist[i_idx]

                for j_idx, j_branch, j_edges in right:
                    d_branch = i_branch + j_branch
                    d_topo = i_edges + j_edges

                    bd_row[j_idx] = d_branch
                    branch_dist[j_idx, i_idx] = d_branch

                    td_row[j_idx] = d_topo
                    topology_dist[j_idx, i_idx] = d_topo

    # Merge all descendant leaf info and return upward
    merged = []
    for lst in child_lists:
        merged.extend(lst)

    return merged


# Run postorder fill
_ = fill_pairwise_distances_postorder(pruned_t)

# Diagonal is already zero, but this makes it explicit
np.fill_diagonal(branch_dist, 0.0)
np.fill_diagonal(topology_dist, 0)

np.save(path_out_dir / "processed_fishtree_branch_distance.npy", branch_dist)
np.save(path_out_dir / "processed_fishtree_topology_distance.npy", topology_dist)

print(f"Saved branch-length distance matrix to: {path_out_dir / 'processed_fishtree_branch_distance.npy'}")
print(f"Saved topology distance matrix to:     {path_out_dir / 'processed_fishtree_topology_distance.npy'}")

Saved branch-length distance matrix to: output\processed_fishtree_branch_distance.npy
Saved topology distance matrix to:     output\processed_fishtree_topology_distance.npy


# Stage 7 — Export the pruned tree as a graph for downstream GNN use

## 21. Convert the pruned tree into an undirected weighted graph

This cell converts the pruned phylogeny into a graph representation that can be used directly by later graph-based models.

The export includes:

- `edge_index`: graph connectivity in a bidirectional edge list format
- `edge_weight`: branch lengths for each edge
- `nodes.csv`: metadata for every node, including which nodes are leaves and which leaf corresponds to which species

This avoids the need to parse Newick files inside later modeling notebooks and provides a clean bridge between the phylogeny and graph neural network workflows.

In [21]:
# ------------------------------------------------------------
# Export pruned tree as undirected graph
# ------------------------------------------------------------
node_to_id = {}
node_rows = []

for node_id, node in enumerate(pruned_t.traverse("preorder")):
    node_to_id[node] = node_id
    is_leaf = node.is_leaf()

    node_rows.append({
        "node_id": node_id,
        "node_name": node.name if is_leaf else f"internal_{node_id}",
        "is_leaf": bool(is_leaf),
        "species": node.name if is_leaf else pd.NA,
    })

edges_src = []
edges_dst = []
edges_w = []

for node in pruned_t.traverse("preorder"):
    u = node_to_id[node]
    for child in node.children:
        v = node_to_id[child]
        w = float(child.dist)

        # undirected: store both directions
        edges_src.extend([u, v])
        edges_dst.extend([v, u])
        edges_w.extend([w, w])

edge_index = np.array([edges_src, edges_dst], dtype=np.int64)
edge_weight = np.array(edges_w, dtype=np.float32)

df_nodes = pd.DataFrame(node_rows)
df_nodes.to_csv(path_out_dir / "processed_fishtree_nodes.csv", index=False)

np.save(path_out_dir / "processed_fishtree_edge_index.npy", edge_index)
np.save(path_out_dir / "processed_fishtree_edge_weight.npy", edge_weight)

print(f"Saved edge_index to:  {path_out_dir / 'processed_fishtree_edge_index.npy'}")
print(f"Saved edge_weight to: {path_out_dir / 'processed_fishtree_edge_weight.npy'}")
print(f"Saved node table to:  {path_out_dir / 'processed_fishtree_nodes.csv'}")

Saved edge_index to:  output\processed_fishtree_edge_index.npy
Saved edge_weight to: output\processed_fishtree_edge_weight.npy
Saved node table to:  output\processed_fishtree_nodes.csv
